# w2 · Sequence preprocessing

This notebook is the sequential pipeline of the KKBox churn data. It runs
independently from the raw archives and produces the two sequence representations
consumed by the LSTM models; the tabular tables live in the w1 tabular pipeline.

**Route:** raw archives → target users → members, transactions and daily listening →
sequence parameters → listening tensor and combined tensor → audited v3 sequence export.

| Product | Shape / grain | Consumer |
|---|---|---|
| `seq_hybrid_*.npy` | users × 90 days × 8 listening channels | listening and hybrid LSTM |
| `seq_lstm_*.npy` | users × 90 days × 20 channels | combined LSTM |
| `metadata.json`, `sequence_audit.json`, `preprocess_params.json` | one file per export | contract and audit |

## 1. Configuration

`KKBOX_SAMPLE_USERS=100` runs a small, isolated export in `export-sequential-v3-sample/`.
The normal run uses all users and writes `export-sequential-v3/`. The competition
cutoffs, 90-day sequence length and channel order are the v3 contract. The full
export needs roughly 10.5 GB of free disk for the four sequence tensors. Polars
streaming keeps the raw logs out of Python memory, while each full listening tensor
is held as an array before its disk copy.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess

import numpy as np
import pandas as pd
import polars as pl


LOG_COLS = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]
LOG_NAMES = {
    "num_25": "25", "num_50": "50", "num_75": "75", "num_985": "985",
    "num_100": "100", "num_unq": "unq", "total_secs": "total_secs",
}

CATEGORICAL = ["city", "registered_via", "last_payment_method_id"]

SEQ_DAYS = 90
SEQ_CHANNELS = LOG_COLS + ["active"]

# v2 labels cover March churn and the v2 submission covers April churn.
# January/February cutoffs belong to train.csv / the original submission, not v2.
CUTOFF = {"train": "2017-02-28", "test": "2017-03-31"}
LABEL_MONTHS = {"train": "2017-03", "test": "2017-04"}
DATASET_VERSION = 3

_TXN_KEYS = ["payment_method_id", "payment_plan_days", "plan_list_price", "actual_amount_paid",
             "is_auto_renew", "transaction_date", "membership_expire_date", "is_cancel"]

# Explicit schemas for the CSV -> parquet conversion (plan 0.5). Smallest types that fit the
# observed ranges (CP-01): city 1-22, registered_via -1-19, bd -7168-2016, plan_days 0-450,
# prices 0-2000, payment_method_id 2-41.
SCHEMA = {
    "train_v2": {"msno": pl.String, "is_churn": pl.Int8},
    "sample_submission_v2": {"msno": pl.String, "is_churn": pl.Int8},
    "members_v3": {"msno": pl.String, "city": pl.Int8, "bd": pl.Int16,
                   "registered_via": pl.Int8, "registration_init_time": pl.Int32,
                   "gender": pl.String},
    "transactions": {"msno": pl.String, "payment_method_id": pl.Int8, "payment_plan_days": pl.Int16,
                     "plan_list_price": pl.Int16, "actual_amount_paid": pl.Int16,
                     "is_auto_renew": pl.Int8, "transaction_date": pl.Int32,
                     "membership_expire_date": pl.Int32, "is_cancel": pl.Int8},
    "transactions_v2": {"msno": pl.String, "payment_method_id": pl.Int8,
                        "payment_plan_days": pl.Int16, "plan_list_price": pl.Int16,
                        "actual_amount_paid": pl.Int16, "is_auto_renew": pl.Int8,
                        "transaction_date": pl.Int32, "membership_expire_date": pl.Int32,
                        "is_cancel": pl.Int8},
    "user_logs": {"msno": pl.String, "date": pl.Int32, "num_25": pl.Int32, "num_50": pl.Int32,
                  "num_75": pl.Int32, "num_985": pl.Int32, "num_100": pl.Int32,
                  "num_unq": pl.Int32, "total_secs": pl.Float64},
    "user_logs_v2": {"msno": pl.String, "date": pl.Int32, "num_25": pl.Int32, "num_50": pl.Int32,
                     "num_75": pl.Int32, "num_985": pl.Int32, "num_100": pl.Int32,
                     "num_unq": pl.Int32, "total_secs": pl.Float64},
}

_COLUMNS = {
    "train_v2": ["msno", "is_churn"],
    "sample_submission_v2": ["msno"],
    "members_v3": ["msno", "city", "registered_via", "bd", "registration_init_time", "gender"],
    "transactions": ["msno"] + _TXN_KEYS,
    "transactions_v2": ["msno"] + _TXN_KEYS,
    "user_logs": ["msno", "date"] + LOG_COLS,
    "user_logs_v2": ["msno", "date"] + LOG_COLS,
}

In [ ]:
import gc
import os
import resource

DATA_DIR = Path('data')
PARQUET_DIR = DATA_DIR / 'parquet'
SAMPLE_USERS = int(os.environ['KKBOX_SAMPLE_USERS']) if os.environ.get('KKBOX_SAMPLE_USERS') else None
if SAMPLE_USERS is not None and SAMPLE_USERS < 1:
    raise ValueError('KKBOX_SAMPLE_USERS must be positive.')
EXPORT_DIR = Path('export-sequential-v3-sample' if SAMPLE_USERS is not None else 'export-sequential-v3')
print('Building member, transaction and listening features from parquet.')
if SAMPLE_USERS is None:
    required_bytes = (970960 + 907471) * SEQ_DAYS * 28 * 2 + 1_000_000_000
    if shutil.disk_usage(EXPORT_DIR.parent).free < required_bytes:
        raise OSError('Insufficient free disk for a full dual sequence export (~10.5 GB).')
print('Export directory:', EXPORT_DIR, '| sample users per split:', SAMPLE_USERS)

## 2. Raw sources and parquet

| CSV archive | Grain / key | Fields used |
|---|---|---|
| `train_v2` | labeled user, `msno` | `is_churn` |
| `sample_submission_v2` | score user, `msno` | identifier only |
| `members_v3` | member, `msno` | city, age, registration source/date |
| `transactions`, `transactions_v2` | payment record, `msno` and transaction fields | plans, amounts, dates, cancellation, renewal |
| `user_logs`, `user_logs_v2` | listening record, `msno` and date | seven listening measures |

The two transaction and listening releases jointly cover the train/test cutoffs.
Conversion selects used columns and assigns compact physical types; it does not
impute or clip values. Existing parquet files are skipped. Extraction handles one
archive at a time and materializes parquet at `sink_parquet`; no conversion occurs
until the following execution cell.

In [ ]:
def window_start(cutoff):
    """First day of the 3-month window that ends with the cutoff month."""
    return (pd.Timestamp(cutoff).to_period("M") - 2).start_time


def sequence_start(cutoff):
    """First of the 90 days that end on the cutoff."""
    return pd.Timestamp(cutoff) - pd.Timedelta(days=SEQ_DAYS - 1)


def _date_int(value):
    return int(pd.Timestamp(value).strftime("%Y%m%d"))


def _date(value):
    return pd.Timestamp(value).date()


def _7z_binary():
    for binary in ("7z", "7za", "7zz"):
        path = shutil.which(binary)
        if path:
            return path
    raise RuntimeError("Converting .7z files needs the `7z` command (p7zip).")

In [ ]:
def convert_raw_to_parquet(data_dir, parquet_dir=None, archives=None):
    """Convert ``data/*.csv.7z`` to ``data/parquet/*.parquet`` (zstd), idempotently.

    Each archive is extracted to ``data/_extract`` one at a time, streamed to parquet and the
    CSV deleted, so the disk peak is one extracted CSV plus the parquet being written.
    """
    data_dir = Path(data_dir)
    parquet_dir = Path(parquet_dir) if parquet_dir else data_dir / "parquet"
    parquet_dir.mkdir(parents=True, exist_ok=True)
    extract_dir = data_dir / "_extract"

    for name in (archives or list(SCHEMA)):
        out = parquet_dir / f"{name}.parquet"
        if out.exists():
            print(f"[skip] {out.name} already exists")
            continue
        archive = data_dir / f"{name}.csv.7z"
        if not archive.exists():
            print(f"[skip] {archive} not found")
            continue

        print(f"[extract] {archive.name}")
        subprocess.run([_7z_binary(), "x", "-y", f"-o{extract_dir}", str(archive)],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        csv = next(path for path in extract_dir.rglob(f"{name}.csv"))
        print(f"[convert] {name}.csv -> {out.name}")
        (pl.scan_csv(str(csv), schema_overrides=SCHEMA[name])
           .select(_COLUMNS[name])
           .sink_parquet(out, compression="zstd"))
        csv.unlink()

        if name.startswith("user_logs"):
            rows = pl.scan_parquet(out).select(pl.len()).collect().item()
            print(f"          {rows:,} rows")

    return parquet_dir

In [ ]:
convert_raw_to_parquet(DATA_DIR)
print('Parquet files:', [p.name for p in sorted(PARQUET_DIR.glob('*.parquet'))])
print('Train schema:', pl.scan_parquet(PARQUET_DIR / 'train_v2.parquet').collect_schema())

## 3. Target users and calendar

| Set | Inclusive daily window | Known through | Target month |
|---|---|---|---|
| Train v2 | 2016-12-01 to 2017-02-28 | 2017-02-28 | March 2017 |
| Test v2 | 2017-01-01 to 2017-03-31 | 2017-03-31 | April 2017 |

Each sequence has exactly 90 **calendar days**; monthly aggregates use calendar
months. `msno` is the source identifier, `uid` is a code shared across train and
test, `row` is the position within one output split, and `day_index` is 0–89 within
that split's window. The same `uid` can have different rows and cutoffs in train
and test. The source file's order determines output rows.

In [ ]:
def load_target_users(parquet_dir):
    """Load train/test users and build the ``msno`` -> ``uid`` union index.

    ``uid`` is assigned over the union of train and test users. ``train_row`` / ``test_row``
    hold the position of the user in the corresponding source CSV (null when absent), because the
    competition files overlap.
    """
    parquet_dir = Path(parquet_dir)
    train = (pl.read_parquet(parquet_dir / "train_v2.parquet", columns=["msno", "is_churn"])
             .with_row_index("row"))
    test = (pl.read_parquet(parquet_dir / "sample_submission_v2.parquet", columns=["msno"])
            .with_row_index("row"))

    union = pl.concat([train.select("msno"), test.select("msno")]).unique(maintain_order=True)
    user_index = (union.with_row_index("uid")
                  .with_columns(pl.col("uid").cast(pl.UInt32)))
    user_index = user_index.join(
        train.select("msno", pl.col("row").alias("train_row")), on="msno", how="left",
        maintain_order="left")
    user_index = user_index.join(
        test.select("msno", pl.col("row").alias("test_row")), on="msno", how="left",
        maintain_order="left")
    return train, test, user_index


def users_with_codes(target_users, user_index):
    """Return ``target_users`` as ``msno`` + ``uid`` + ``row`` in the original order."""
    return (target_users.select("msno", "row")
            .join(user_index.select("msno", "uid"), on="msno", how="left", maintain_order="left"))

In [ ]:
def _hash_msno(df):
    digest = hashlib.sha1()
    for value in df["msno"].to_list():
        digest.update(str(value).encode())
        digest.update(b"\n")
    return digest.hexdigest()




In [ ]:
train_users, test_users, user_index = load_target_users(PARQUET_DIR)
if SAMPLE_USERS is not None:
    train_users = train_users.head(SAMPLE_USERS)
    test_users = test_users.head(SAMPLE_USERS)
    selected = pl.concat([train_users.select('msno'), test_users.select('msno')]).unique()
    user_index = user_index.join(selected, on='msno', how='semi')
users_train = users_with_codes(train_users, user_index)
users_test = users_with_codes(test_users, user_index)
print('Users:', train_users.height, test_users.height, '| unique uid:', user_index.height)
print(users_train.head(3))
for split, cutoff in CUTOFF.items():
    print(split, 'monthly start:', window_start(cutoff).date(),
          'sequence start:', sequence_start(cutoff).date(), 'cutoff:', cutoff)
example = pl.DataFrame({'msno': ['A', 'B', 'C'], 'uid': [0, 1, 2],
                        'train_row': [0, 1, None], 'test_row': [2, None, 0]})
print('Example of split-specific rows:', example)

## 4. Members

Members are filtered to target `uid`s before building features. Valid age is
1–100; other ages receive the mean valid train age and `bd_missing=1`.
Negative `registered_via` receives the train mode. Registration dates yield
tenure, year and month at each cutoff. `gender` is loaded by the raw schema but
excluded from the existing feature catalog. These imputation values are fitted
once using the competition train population.

In [ ]:
def scan_members(parquet_dir, user_index):
    """Lazy members table filtered to target users, keyed by ``uid``."""
    return (pl.scan_parquet(Path(parquet_dir) / "members_v3.parquet")
            .join(user_index.lazy().select("msno", "uid"), on="msno", how="inner")
            .select("uid", "city", "registered_via", "bd", "registration_init_time"))


def scan_transactions(parquet_dir, user_index, max_date):
    """Lazy transactions (both files, deduplicated) up to ``max_date``, keyed by ``uid``."""
    parquet_dir = Path(parquet_dir)
    frames = [pl.scan_parquet(parquet_dir / "transactions.parquet"),
              pl.scan_parquet(parquet_dir / "transactions_v2.parquet")]
    return (pl.concat(frames)
            .with_row_index("_ord")
            .join(user_index.lazy().select("msno", "uid"), on="msno", how="inner",
                  maintain_order="left")
            .filter(pl.col("transaction_date") <= _date_int(max_date))
            .unique(subset=["msno", *_TXN_KEYS], keep="first", maintain_order=True)
            .select("uid", "_ord", *_TXN_KEYS))

## 5. Transactions and deduplication

Both releases are concatenated, filtered to target users and cutoff, then exact
duplicates are removed using `msno` plus all eight transaction fields. The first
copy is kept. Two records on the same day with different values remain separate.
`_ord` records stable source order; it is a tie break, **not an intraday time**.
Conflicting cancellation, renewal or expiry values on one day mark ambiguous
state. `is_cancel=1` is a payment record field, not the churn target label.

The sequence preflight rejects missing/invalid required transaction dates, plan
days and binary flags; missing expiry has a separate mask. It counts negative
amounts and invalid expiry values for the audit. A future transaction never
updates an earlier day's state.

In [ ]:
members = scan_members(PARQUET_DIR, user_index).collect(engine='streaming')
transactions = scan_transactions(PARQUET_DIR, user_index, CUTOFF['test']).collect(engine='streaming')
print('Members:', members.shape, '| transactions:', transactions.shape)
print(transactions.select('uid', '_ord', 'transaction_date', 'actual_amount_paid', 'is_cancel').head(3))

In [ ]:
# Two different payments on one date survive; an exact second copy does not.
first_transaction = transactions.row(0, named=True)
duplicate_example = pl.DataFrame([
    first_transaction,
    dict(first_transaction),
    {**first_transaction, 'actual_amount_paid': first_transaction['actual_amount_paid'] + 1},
])
kept_example = duplicate_example.unique(subset=['uid', *_TXN_KEYS], keep='first')
print('Same-day source rows:', duplicate_example.height, '| after exact deduplication:', kept_example.height)
print(kept_example.select('uid', 'transaction_date', 'actual_amount_paid'))

## 7. Daily listening

The two log releases are filtered to the union of needed dates and target `uid`s,
then grouped by `(uid, date)`. Negative counts are clipped to zero; nonfinite
seconds become zero, and daily sums are kept in compact types. `rows` counts raw
records for the daily group. A date with a record is active even if every listening
measure is zero. The lazy Polars scan runs only when `collect(engine="streaming")`
materializes the grouped user-day table; previews below reuse that result.

In [ ]:
def build_user_day_logs(parquet_dir, user_index, start, end):
    """One row per ``(uid, date)`` with the summed listening columns and the raw row count.

    ``start`` / ``end`` are ``YYYYMMDD`` ints. This is the single intermediate table that feeds
    both the monthly aggregates and the daily sequences.
    """
    parquet_dir = Path(parquet_dir)
    paths = [parquet_dir / "user_logs.parquet"]
    if _date_int(end) >= 20170301:
        march = parquet_dir / "user_logs_v2.parquet"
        if not march.exists():
            raise FileNotFoundError("The March cutoff requires user_logs_v2.parquet.")
        paths.append(march)
    lf = (pl.concat([pl.scan_parquet(path) for path in paths])
          .filter((pl.col("date") >= _date_int(start)) & (pl.col("date") <= _date_int(end)))
          .join(user_index.lazy().select("msno", "uid"), on="msno", how="inner")
          .group_by("uid", "date")
          .agg([pl.col(c).cast(pl.Int64).clip(lower_bound=0).sum()
                .clip(upper_bound=np.iinfo(np.int32).max).cast(pl.Int32)
                for c in LOG_COLS if c != "total_secs"]
               + [pl.when(pl.col("total_secs").is_finite())
                    .then(pl.col("total_secs")).otherwise(0).clip(lower_bound=0)
                    .sum().cast(pl.Float32), pl.len().alias("rows")]))
    return lf.collect(engine="streaming")

In [ ]:
LOG_START = min(window_start(CUTOFF['train']), sequence_start(CUTOFF['train']))
user_day = build_user_day_logs(PARQUET_DIR, user_index, LOG_START, CUTOFF['test'])
print('User-day rows:', user_day.height)
print(user_day.head(3))

## 8. Listening features and parameters

Valid train ages/modes and the 99.9th percentile clipping values come from the
competition train window. Monthly listening groups by user and calendar month;
the monthly mean is clipped at the daily-derived threshold, and the sum is
rebuilt from that mean times the record count. Ratios use total songs where
positive. `last`, `last_second` and `last_third` mean the most recent **active**
months among the three eligible months; they need not be consecutive.

`days_since_last_log` is relative to the split cutoff. A user without logs gets
`has_listening_logs=0` and an out-of-window recency sentinel. Daily v3 tensors
remain unscaled log1p values. Model experiments fit their own scales using fit
data. The aggregate clipping parameters here are not refitted inside each model
fold; that inherited limitation remains in this readability refactor.

In [ ]:
def fit_preprocess_params(members, user_day, train_users, train_cutoff):
    """Learn imputation, clipping and sequence standardization from the training window only."""
    members_train = members.join(train_users.select("uid"), on="uid", how="semi")

    age_mean = float(
        members_train.filter((pl.col("bd") > 0) & (pl.col("bd") <= 100))
                     .select(pl.col("bd").mean()).item())
    via_mode = int(
        members_train.filter(pl.col("registered_via") >= 0)
                     .select(pl.col("registered_via").mode().first()).item())

    start, cutoff = _date_int(sequence_start(train_cutoff)), _date_int(train_cutoff)
    window = (user_day.join(train_users.select("uid"), on="uid", how="semi")
                       .filter((pl.col("date") >= start) & (pl.col("date") <= cutoff)))
    if window.is_empty():
        raise ValueError("No training listening records inside the sequence window.")

    clip, mean, std = {}, np.empty(len(LOG_COLS)), np.empty(len(LOG_COLS))
    for i, col in enumerate(LOG_COLS):
        upper = window.select(pl.col(col).quantile(0.999, interpolation="linear")).item()
        cell = window.select(pl.col(col).clip(upper_bound=upper).log1p()).to_series()
        clip[col] = float(upper)
        mean[i] = float(cell.mean())
        std[i] = float(cell.std(ddof=0))
    if not np.isfinite(mean).all() or not np.isfinite(std).all():
        raise ValueError("Non-finite daily sequence preprocessing parameters.")
    std[std < 1e-6] = 1.0

    return {
        "age_mean": age_mean,
        "registered_via_mode": via_mode,
        "log_clip_p999": clip,
        "seq_days": int(SEQ_DAYS),
        "seq_mean": mean.tolist(),
        "seq_std": std.tolist(),
    }


def save_preprocess_params(params, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(params, f, indent=2)


def load_preprocess_params(path):
    with open(path) as f:
        return json.load(f)

In [ ]:
params = fit_preprocess_params(members, user_day, users_train, CUTOFF['train'])
print('Age mean:', round(params['age_mean'], 2),
      '| registered-via mode:', params['registered_via_mode'])
print('Daily clipping limits:', params['log_clip_p999'])

## 10. Listening sequence

Each grouped user-day joins its split-specific `row`; `day_index` is the date
offset from the 90-day start. Seven nonnegative measures become `log1p(value)`
and are placed at `[row, day_index, channel]`; channel 8 is `active=1` when a log
row exists. Unobserved days remain all zero. Duplicate positions and nonfinite
values fail validation.

For a five-day teaching window with logs on days 0 and 3, the active channel is
`[1, 0, 0, 1, 0]`; a zero-count log on day 3 still has `active=1`. The production
array always has 90 days. This sequence uses no model-fitted standardization.

In [ ]:
def build_log_sequences(user_day, users, cutoff, params, standardize=True):
    """Calendar-day tensor in float16, aligned by users.row.

    ``standardize=False`` exports unscaled log1p counts so each modelling split can
    learn clipping and scaling on its own fit subset. The legacy default retains
    the original training-window clipping/standardization contract.
    """
    if users["uid"].n_unique() != users.height:
        raise ValueError("Sequence users must have unique uid values.")
    if not np.array_equal(np.sort(users["row"].to_numpy()), np.arange(users.height)):
        raise ValueError("Sequence row values must be a permutation of 0..n_users-1.")
    start, cutoff_int = sequence_start(cutoff), _date_int(cutoff)
    mean = np.asarray(params["seq_mean"], dtype=np.float32)
    std = np.asarray(params["seq_std"], dtype=np.float32)
    clip = params["log_clip_p999"]

    uw = (user_day.join(users.select("uid"), on="uid", how="semi")
          .filter((pl.col("date") >= _date_int(start)) & (pl.col("date") <= cutoff_int))
          .join(users.select("uid", "row"), on="uid", how="inner"))
    if standardize:
        uw = uw.with_columns([
            pl.col(c).clip(lower_bound=0, upper_bound=clip[c]).log1p().alias(c)
            for c in LOG_COLS])
        uw = uw.with_columns([
            ((pl.col(c) - mean[i]) / std[i]).cast(pl.Float32).alias(c)
            for i, c in enumerate(LOG_COLS)])
    else:
        uw = uw.with_columns([
            pl.col(c).clip(lower_bound=0).log1p().cast(pl.Float32).alias(c)
            for c in LOG_COLS])
    uw = uw.with_columns(
        (pl.col("date").cast(pl.String).str.to_date("%Y%m%d") - pl.lit(_date(start)))
        .dt.total_days().cast(pl.Int32).alias("day"))

    n = users.height
    seq = np.zeros((n, SEQ_DAYS, len(SEQ_CHANNELS)), dtype=np.float16)
    rows = uw["row"].to_numpy()
    days = uw["day"].to_numpy()
    positions = rows.astype(np.int64) * SEQ_DAYS + days
    if len(np.unique(positions)) != len(positions):
        raise ValueError("Daily sequences require one row per user and calendar day.")
    values = uw.select(LOG_COLS).to_numpy().astype(np.float16)
    if not np.isfinite(values).all():
        raise ValueError("Daily sequence values are not finite after preprocessing.")
    seq[rows, days, :len(LOG_COLS)] = values
    seq[rows, days, -1] = 1
    return seq


def audit_log_sequences(seq, encoding="log1p", chunk_size=4096):
    """Validate sequence values in bounded memory and summarize calendar coverage."""
    if seq.ndim != 3 or seq.shape[1:] != (SEQ_DAYS, len(SEQ_CHANNELS)):
        raise ValueError("Unexpected daily sequence dimensions.")
    empty, total_days = 0, 0
    by_day = np.zeros(SEQ_DAYS, dtype=np.int64)
    for start in range(0, len(seq), chunk_size):
        batch = np.asarray(seq[start:start + chunk_size])
        if not np.isfinite(batch).all():
            raise ValueError("Non-finite sequence values.")
        active = batch[:, :, -1]
        if not np.isin(active, [0, 1]).all():
            raise ValueError("Sequence active flag must be binary.")
        if np.any(batch[:, :, :-1][active == 0] != 0):
            raise ValueError("Days without logs must have zero listening channels.")
        if encoding == "log1p" and (batch[:, :, :-1] < 0).any():
            raise ValueError("Unscaled log1p listening channels must be nonnegative.")
        counts = active.sum(axis=1, dtype=np.int64)
        empty += int((counts == 0).sum())
        total_days += int(counts.sum())
        by_day += active.sum(axis=0, dtype=np.int64)
    return {"n_users": len(seq), "users_without_logs": empty,
            "share_without_logs": empty / max(len(seq), 1),
            "mean_days_with_logs": total_days / max(len(seq), 1),
            "users_with_logs_by_day": by_day.tolist()}

In [ ]:
toy = np.zeros((1, 5, len(SEQ_CHANNELS)), dtype=np.float16)
toy[0, [0, 3], -1] = 1
toy[0, 0, 0] = np.log1p(4)
print('Five-day active:', toy[0, :, -1].tolist(),
      '| first channel:', toy[0, :, 0].tolist())

## 11. Transaction events and state

The transaction tensor has twelve channels, appended after the eight listening
channels. The first five are **events** on the transaction day. The other seven
describe the last state known by the end of that day and carry forward until a
new event. A pre-window transaction initializes state but creates no day-zero
event. A future event never fills earlier days backward.

| Channel | Meaning, source and encoding | Validity / missing |
|---|---|---|
| `tx_count` | daily record count, log1p | zero without event |
| `tx_paid_sum` | daily NTD sum, signed-log1p | zero without event |
| `tx_plan_days_sum` | daily plan-day sum, log1p | zero without event |
| `tx_cancel_count` | daily cancel count, log1p | zero without event |
| `tx_auto_renew_count` | daily renew count, log1p | zero without event |
| `tx_state_known` | any transaction seen, binary | zero without history |
| `tx_last_is_cancel` | last daily record, binary | masked by state known |
| `tx_last_is_auto_renew` | last daily record, binary | masked by state known |
| `tx_days_since_last` | calendar days since last record, log1p, cap 3650 | masked by state known |
| `tx_days_to_expire` | expiry minus day, signed-log1p, cap ±3650 | masked by expiry known |
| `tx_expiry_known` | valid last expiry, binary | zero if absent |
| `tx_state_ambiguous` | conflicting same-day states, binary | masked by state known |

Signed-log1p maps `5 → log(6)`, `−5 → −log(6)`, `0 → 0`. An expiry date in the
future is usable if a past payment already recorded it. State remains unknown
for users without history; zero-valued event channels also occur on no-event
days, so `tx_count` distinguishes those cases. `sequence_channel_specs()` is the
machine-readable channel dictionary.

In [ ]:
TX_CHANNELS = [
    "tx_count", "tx_paid_sum", "tx_plan_days_sum", "tx_cancel_count",
    "tx_auto_renew_count", "tx_state_known", "tx_last_is_cancel",
    "tx_last_is_auto_renew", "tx_days_since_last", "tx_days_to_expire",
    "tx_expiry_known", "tx_state_ambiguous",
]
LSTM_CHANNELS = SEQ_CHANNELS + TX_CHANNELS

In [ ]:
def sequence_channel_specs():
    """Ordered, machine-readable transforms and validity masks for v3."""
    specs = []
    for name in LSTM_CHANNELS:
        flag = name in {"active", "tx_state_known", "tx_last_is_cancel",
                        "tx_last_is_auto_renew", "tx_expiry_known", "tx_state_ambiguous"}
        signed = name in {"tx_paid_sum", "tx_days_to_expire"}
        if name in LOG_COLS:
            mask, group, unit = "active", "listening", "seconds" if name == "total_secs" else "count"
        elif name == "active":
            mask, group, unit = None, "listening", "flag"
        elif name in TX_CHANNELS[:5]:
            mask, group, unit = "tx_count", "event", "NTD" if name == "tx_paid_sum" else "days" if name == "tx_plan_days_sum" else "count"
        else:
            mask, group, unit = "tx_expiry_known" if name == "tx_days_to_expire" else "tx_state_known", "state", "flag" if flag else "days"
        specs.append({"name": name, "encoding": "flag" if flag else "signed_log1p" if signed else "log1p",
                      "group": group, "mask": mask, "units": unit,
                      "fixed_limits": [-3650, 3650] if name == "tx_days_to_expire" else [0, 3650] if name == "tx_days_since_last" else None})
    return specs

In [ ]:
def prepare_transaction_days(trans, users, cutoff):
    """Sparse daily events plus the last pre-window state; no future fill.

    _ord is only a deterministic source-order tie-break, not a timestamp.
    """
    if users["uid"].n_unique() != users.height or not np.array_equal(
            np.sort(users["row"].to_numpy()), np.arange(users.height)):
        raise ValueError("Users require unique uid and a row permutation.")
    lf = trans.lazy() if isinstance(trans, pl.DataFrame) else trans
    t = (lf.join(users.select("uid", "row").lazy(), on="uid", how="inner")
         .unique(subset=["uid", *_TXN_KEYS], keep="first", maintain_order=True)
         .with_columns(
             pl.col("transaction_date").cast(pl.String).str.to_date("%Y%m%d", strict=False).alias("td"),
             pl.col("membership_expire_date").cast(pl.String).str.to_date("%Y%m%d", strict=False).alias("expiry"))
         .filter((pl.col("transaction_date") <= _date_int(cutoff)) | pl.col("td").is_null()))
    required = ["transaction_date", "payment_plan_days", "actual_amount_paid", "is_cancel", "is_auto_renew", "_ord"]
    invalid = (pl.any_horizontal([pl.col(c).is_null() for c in required]) |
               pl.col("td").is_null() | (pl.col("payment_plan_days") < 0) |
               ~pl.col("is_cancel").is_in([0, 1]) | ~pl.col("is_auto_renew").is_in([0, 1]))
    preflight = t.select(
        pl.len().alias("n_transactions"), invalid.sum().alias("invalid_required"),
        (pl.col("actual_amount_paid") < 0).sum().alias("negative_amounts"),
        pl.col("expiry").is_null().sum().alias("invalid_expiry")).collect().row(0, named=True)
    if preflight["invalid_required"]:
        raise ValueError(f"Invalid transaction fields: {preflight}")
    daily = (t.group_by("row", "td").agg(
        pl.len().alias("tx_count"),
        pl.col("actual_amount_paid").cast(pl.Int64).sum().alias("tx_paid_sum"),
        pl.col("payment_plan_days").cast(pl.Int64).sum().alias("tx_plan_days_sum"),
        pl.col("is_cancel").cast(pl.Int64).sum().alias("tx_cancel_count"),
        pl.col("is_auto_renew").cast(pl.Int64).sum().alias("tx_auto_renew_count"),
        pl.col("is_cancel").sort_by("_ord").last().alias("cancel"),
        pl.col("is_auto_renew").sort_by("_ord").last().alias("renew"),
        pl.col("expiry").sort_by("_ord").last().alias("expiry"),
        ((pl.col("is_cancel").n_unique() > 1) | (pl.col("is_auto_renew").n_unique() > 1) |
         (pl.col("expiry").n_unique() > 1)).cast(pl.Int8).alias("ambiguous"))
        .collect(engine="streaming"))
    start = _date(sequence_start(cutoff))
    before = daily.filter(pl.col("td") < start).sort("row", "td").group_by("row", maintain_order=True).tail(1)
    inside = daily.filter(pl.col("td") >= start)
    days = (pl.concat([before, inside]).with_columns(
        pl.col("td").cast(pl.Int32).alias("ordinal"),
        pl.col("expiry").cast(pl.Int32).alias("expiry_ordinal"))
        .sort("row", "td"))
    preflight.update({"daily_rows_retained": days.height,
                      "multiple_event_days": inside.filter(pl.col("tx_count") > 1).height,
                      "ambiguous_event_days": inside.filter(pl.col("ambiguous") == 1).height})
    return days, preflight

The daily table keeps the latest pre-window row and all in-window rows.
Inside each user block, slot zero stores pre-window state; slots 1–90 are real
days. `maximum.accumulate` carries the latest event index forward. Index `−1`
means no known state, and a safe index is used only under the known-state mask.
The float32 block is cast to float16 and written to a `.npy` memory map, so the
full combined tensor need not reside in RAM.

In [ ]:
def build_lstm_sequences(trans, users, cutoff, listening, output_path=None, chunk_size=4096):
    """Build bounded-memory (users, 90, 20) arrays, optionally directly to .npy."""
    if listening.shape != (users.height, SEQ_DAYS, len(SEQ_CHANNELS)):
        raise ValueError("Listening shape does not match users.")
    days, preflight = prepare_transaction_days(trans, users, cutoff)
    names = ["row", "ordinal", "expiry_ordinal", *TX_CHANNELS[:5], "cancel", "renew", "ambiguous"]
    daily_values = {c: days[c].to_numpy() for c in names}
    transaction_rows = daily_values["row"]
    first_day_ordinal = int(np.datetime64(_date(sequence_start(cutoff)), "D").astype(np.int64))
    shape = (users.height, SEQ_DAYS, len(LSTM_CHANNELS))
    if output_path is None:
        result = np.zeros(shape, dtype=np.float16)
    else:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        result = np.lib.format.open_memmap(output_path, mode="w+", dtype=np.float16, shape=shape)
    listening_channels = len(SEQ_CHANNELS)
    tx_channel = {name: listening_channels + index for index, name in enumerate(TX_CHANNELS)}
    limited_since = limited_expiry = 0
    for block_start in range(0, users.height, chunk_size):
        block_end = min(block_start + chunk_size, users.height)
        first_record, after_last_record = np.searchsorted(transaction_rows, [block_start, block_end])
        user_rows = transaction_rows[first_record:after_last_record].astype(np.int64) - block_start
        relative_days = daily_values["ordinal"][first_record:after_last_record].astype(np.int64) - first_day_ordinal
        # Slot zero holds pre-window state; actual days occupy slots 1..90.
        state_index = np.full((block_end-block_start, SEQ_DAYS+1), -1, dtype=np.int64)
        event_indices = np.arange(first_record, after_last_record)
        state_index[user_rows, np.maximum(relative_days+1, 0)] = event_indices
        latest_transaction_index = np.maximum.accumulate(state_index, axis=1)[:, 1:]
        state_known = latest_transaction_index >= 0
        safe_latest_index = np.maximum(latest_transaction_index, 0)
        block_values = np.zeros((block_end-block_start, SEQ_DAYS, len(LSTM_CHANNELS)), dtype=np.float32)
        block_values[:, :, :listening_channels] = listening[block_start:block_end]
        inside_window = relative_days >= 0
        for c in TX_CHANNELS[:5]:
            values = daily_values[c][first_record:after_last_record][inside_window].astype(np.float64)
            encoded = np.sign(values) * np.log1p(np.abs(values))
            block_values[user_rows[inside_window], relative_days[inside_window], tx_channel[c]] = encoded
        block_values[:, :, tx_channel["tx_state_known"]] = state_known
        if len(transaction_rows):
            for dest, source in [(tx_channel["tx_last_is_cancel"], "cancel"),
                                 (tx_channel["tx_last_is_auto_renew"], "renew"),
                                 (tx_channel["tx_state_ambiguous"], "ambiguous")]:
                block_values[:, :, dest] = np.where(state_known, daily_values[source][safe_latest_index], 0)
            calendar_ordinals = first_day_ordinal + np.arange(SEQ_DAYS)[None, :]
            days_since_last = calendar_ordinals - daily_values["ordinal"][safe_latest_index]
            expiry_ordinals = daily_values["expiry_ordinal"][safe_latest_index]
            expiry_known = state_known & np.isfinite(expiry_ordinals)
            days_to_expire = np.where(expiry_known, expiry_ordinals - calendar_ordinals, 0)
            limited_since += int(((days_since_last > 3650) & state_known).sum())
            limited_expiry += int(((np.abs(days_to_expire) > 3650) & expiry_known).sum())
            block_values[:, :, tx_channel["tx_days_since_last"]] = np.where(state_known, np.log1p(np.clip(days_since_last, 0, 3650)), 0)
            days_to_expire = np.clip(days_to_expire, -3650, 3650)
            block_values[:, :, tx_channel["tx_days_to_expire"]] = np.sign(days_to_expire) * np.log1p(np.abs(days_to_expire))
            block_values[:, :, tx_channel["tx_expiry_known"]] = expiry_known
        if not np.isfinite(block_values).all():
            raise ValueError("Non-finite combined sequence values.")
        result[block_start:block_end] = block_values.astype(np.float16)
    if isinstance(result, np.memmap):
        result.flush()
    preflight.update({"limited_days_since_last": limited_since, "limited_days_to_expire": limited_expiry})
    return result, preflight

In [ ]:
def audit_lstm_sequences(seq, listening, chunk_size=4096):
    if seq.shape != (len(listening), SEQ_DAYS, len(LSTM_CHANNELS)):
        raise ValueError("Unexpected combined sequence shape.")
    stats = {"n_users": len(seq), "users_without_transactions": 0,
             "days_with_transactions": 0, "ambiguous_state_days": 0}
    listening_channels = len(SEQ_CHANNELS)
    channel = {name: listening_channels + index for index, name in enumerate(TX_CHANNELS)}
    flags = [channel[name] for name in ("tx_state_known", "tx_last_is_cancel",
                                       "tx_last_is_auto_renew", "tx_expiry_known",
                                       "tx_state_ambiguous")]
    for lo in range(0, len(seq), chunk_size):
        x = np.asarray(seq[lo:lo+chunk_size])
        if not np.isfinite(x).all() or not np.isin(x[:, :, flags], [0, 1]).all():
            raise ValueError("Invalid combined values or flags.")
        if not np.array_equal(x[:, :, :listening_channels], listening[lo:lo+chunk_size]):
            raise ValueError("Combined listening is not aligned with hybrid sequence.")
        if (x[:, :, [channel[name] for name in ("tx_count", "tx_plan_days_sum",
                                               "tx_cancel_count", "tx_auto_renew_count",
                                               "tx_days_since_last")]] < 0).any():
            raise ValueError("Negative counts or recency.")
        events = x[:, :, channel["tx_count"]] > 0
        known = x[:, :, channel["tx_state_known"]] > 0
        if np.any(x[:, :, listening_channels:channel["tx_state_known"]][~events] != 0) or np.any(x[:, :, channel["tx_last_is_cancel"]:][~known] != 0):
            raise ValueError("Invalid missing-event/state masks.")
        if np.any(x[:, :, channel["tx_days_to_expire"]][x[:, :, channel["tx_expiry_known"]] == 0] != 0):
            raise ValueError("Invalid missing-expiry mask.")
        stats["users_without_transactions"] += int((~known.any(axis=1)).sum())
        stats["days_with_transactions"] += int(events.sum())
        stats["ambiguous_state_days"] += int(x[:, :, channel["tx_state_ambiguous"]].sum())
    return stats

In [ ]:
# Small causal fixture: prior payment, in-window event, and a future event.
toy_users = pl.DataFrame({'uid': [1], 'row': [0]})
toy_transactions = pl.DataFrame([
    {'uid': 1, '_ord': 0, 'payment_method_id': 1, 'payment_plan_days': 30,
     'plan_list_price': 100, 'actual_amount_paid': 100, 'is_auto_renew': 1,
     'transaction_date': 20161130, 'membership_expire_date': 20161231, 'is_cancel': 0},
    {'uid': 1, '_ord': 1, 'payment_method_id': 1, 'payment_plan_days': 30,
     'plan_list_price': 100, 'actual_amount_paid': 100, 'is_auto_renew': 1,
     'transaction_date': 20161203, 'membership_expire_date': 20170102, 'is_cancel': 0},
])
toy_listening = np.zeros((1, SEQ_DAYS, len(SEQ_CHANNELS)), dtype=np.float16)
toy_combined, _ = build_lstm_sequences(toy_transactions, toy_users,
                                       CUTOFF['train'], toy_listening)
print('Day 0–3: event, state known, days since last:')
print(toy_combined[0, :4, [8, 13, 16]].T)
future = toy_transactions.vstack(toy_transactions.tail(1).with_columns(
    pl.lit(20170201, dtype=pl.Int64).alias('transaction_date'),
    pl.lit(2, dtype=pl.Int64).alias('_ord')))
with_future, _ = build_lstm_sequences(future, toy_users, CUTOFF['train'], toy_listening)
assert np.array_equal(toy_combined[0, :62], with_future[0, :62])
print('Adding a February event leaves all preceding days unchanged.')

In [ ]:
# Slow reference for this one-user fixture; production uses block-wise NumPy.
transaction_days, _ = prepare_transaction_days(toy_transactions, toy_users, CUTOFF['train'])
records = transaction_days.to_dicts()
first_ordinal = int(np.datetime64(_date(sequence_start(CUTOFF['train'])), 'D').astype(np.int64))
reference = np.zeros((SEQ_DAYS, len(TX_CHANNELS)), dtype=np.float32)
latest = None
for record in records:
    if record['ordinal'] < first_ordinal:
        latest = record
for day_index in range(SEQ_DAYS):
    ordinal = first_ordinal + day_index
    today = next((record for record in records if record['ordinal'] == ordinal), None)
    if today is not None:
        for channel_index, name in enumerate(TX_CHANNELS[:5]):
            value = today[name]
            reference[day_index, channel_index] = np.sign(value) * np.log1p(abs(value))
        latest = today
    if latest is None:
        continue
    reference[day_index, TX_CHANNELS.index('tx_state_known')] = 1
    reference[day_index, TX_CHANNELS.index('tx_last_is_cancel')] = latest['cancel']
    reference[day_index, TX_CHANNELS.index('tx_last_is_auto_renew')] = latest['renew']
    elapsed = min(max(ordinal - latest['ordinal'], 0), 3650)
    reference[day_index, TX_CHANNELS.index('tx_days_since_last')] = np.log1p(elapsed)
    expiry = latest['expiry_ordinal']
    if expiry is not None:
        remaining = np.clip(expiry - ordinal, -3650, 3650)
        reference[day_index, TX_CHANNELS.index('tx_days_to_expire')] = (
            np.sign(remaining) * np.log1p(abs(remaining)))
        reference[day_index, TX_CHANNELS.index('tx_expiry_known')] = 1
    reference[day_index, TX_CHANNELS.index('tx_state_ambiguous')] = latest['ambiguous']
np.testing.assert_array_equal(reference.astype(np.float16), toy_combined[0, :, len(SEQ_CHANNELS):])
print('Reference loop matches every transaction channel for the causal fixture.')

## 12. Combination and block writing

The combined shape is `(users, 90, 20)`: channels 0–7 are the exact listening
tensor and 8–19 are transaction events/state for the **same** `row` and day.
Static tabular features are not repeated as fictional daily history. The writer
uses float16 on disk and user blocks of 4096; transaction states are assembled in
float32 per block. The checks below verify row order and exact listening equality.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(EXPORT_DIR / 'metadata.json').unlink(missing_ok=True)
hybrid, lstm, sequence_audit = {}, {}, {}
for split, user_rows in [('train', users_train), ('test', users_test)]:
    listening = build_log_sequences(user_day, user_rows, CUTOFF[split], params,
                                    standardize=False)
    hybrid_path = EXPORT_DIR / f'seq_hybrid_{split}.npy'
    stored = np.lib.format.open_memmap(hybrid_path, mode='w+', dtype=np.float16,
                                        shape=listening.shape)
    for block_start in range(0, len(listening), 4096):
        block_end = min(block_start + 4096, len(listening))
        stored[block_start:block_end] = listening[block_start:block_end]
    stored.flush()
    hybrid[split] = stored
    lstm[split], tx_preflight = build_lstm_sequences(
        transactions, user_rows, CUTOFF[split], stored,
        output_path=EXPORT_DIR / f'seq_lstm_{split}.npy')
    assert np.array_equal(lstm[split][:, :, :len(SEQ_CHANNELS)], stored)
    sequence_audit[split] = {
        'hybrid': audit_log_sequences(stored),
        'lstm': audit_lstm_sequences(lstm[split], stored),
        'transaction_preflight': tx_preflight,
        'start': str(sequence_start(CUTOFF[split]).date()), 'cutoff': CUTOFF[split],
    }
    print(split, 'shapes:', stored.shape, lstm[split].shape,
          '| no-log users:', sequence_audit[split]['hybrid']['users_without_logs'],
          '| ambiguous days:', tx_preflight['ambiguous_event_days'])
(EXPORT_DIR / 'sequence_audit.json').write_text(json.dumps(sequence_audit, indent=2))

## 13. Audit and export

The listening audit checks shapes, finite values, binary `active`, and zeros
where no log exists. The combined audit checks its flags, missing-event/state
masks, nonnegative counts/recency and exact equality of the first eight channels.
The preflight reports users without transactions, ambiguous days and capped
recencies. The row hash verifies identifier order; it does not independently
prove that a value belongs to the correct user, hence the explicit row and
tensor checks. Metadata is published last, after all arrays and tables exist.

In [ ]:
def export_sequence_dataset(export_dir, hybrid, lstm, users, metadata):
    """Validate dual sequence arrays and publish their contract; tables stay in w1."""
    root = Path(export_dir)
    root.mkdir(parents=True, exist_ok=True)
    # Invalidate a previous manifest before overwriting any part of this export.
    (root / "metadata.json").unlink(missing_ok=True)
    meta = dict(metadata)
    meta["dataset_version"] = 3
    meta["sequences"] = {}
    audit = {}
    for split, user_rows in users.items():
        if user_rows["uid"].n_unique() != user_rows.height or len(hybrid[split]) != user_rows.height:
            raise ValueError("Sequence shape/users do not match table.")
        audit[split] = {"hybrid": audit_log_sequences(hybrid[split]),
                        "lstm": audit_lstm_sequences(lstm[split], hybrid[split])}
        meta[f"n_{split}"] = user_rows.height
        meta[f"hash_{split}"] = _hash_msno(user_rows)
    for family, arrays, channels in [("hybrid", hybrid, SEQ_CHANNELS), ("lstm", lstm, LSTM_CHANNELS)]:
        spec = {"channels": channels, "channel_specs": sequence_channel_specs()[:len(channels)],
                "dtype": "float16", "order": "oldest_to_newest_calendar_day"}
        for split, array in arrays.items():
            path = root / f"seq_{family}_{split}.npy"
            if array.dtype != np.float16:
                raise ValueError("Dual export requires float16 arrays.")
            existing = getattr(array, "filename", None)
            if existing is None or Path(existing).resolve() != path.resolve():
                np.save(path, array)
            elif isinstance(array, np.memmap):
                array.flush()
            spec[split] = {"file": path.name, "shape": list(array.shape),
                           "row_hash": meta[f"hash_{split}"]}
        meta["sequences"][family] = spec
    meta["transaction_sources"] = ["transactions.parquet", "transactions_v2.parquet"]
    meta["transaction_tie_policy"] = {"version": 1, "order": ["transaction_date", "_ord"],
                                      "meaning": "deterministic source order; intraday chronology unknown"}
    meta["generated_at"] = pd.Timestamp.now().isoformat(timespec="seconds")
    meta["sequence_audit"] = audit
    temporary = root / "metadata.json.tmp"
    temporary.write_text(json.dumps(meta, indent=2))
    temporary.replace(root / "metadata.json")
    return meta

In [ ]:
save_preprocess_params(params, EXPORT_DIR / 'preprocess_params.json')
metadata = export_sequence_dataset(EXPORT_DIR, hybrid, lstm, {
    'train': users_train, 'test': users_test,
}, {
    'cutoffs': CUTOFF, 'label_months': LABEL_MONTHS,
    'seq_days': SEQ_DAYS, 'log_sources': ['user_logs.parquet', 'user_logs_v2.parquet'],
    'windows': {split: {'start': str(sequence_start(date).date()), 'cutoff': date}
                for split, date in CUTOFF.items()},
    'preprocess_params': params,
})
for split, user_rows in [('train', users_train), ('test', users_test)]:
    assert metadata[f'hash_{split}'] == _hash_msno(user_rows)
    assert metadata['sequences']['lstm'][split]['row_hash'] == _hash_msno(user_rows)
print('Export:', EXPORT_DIR.resolve())
for path in sorted(EXPORT_DIR.iterdir()):
    if path.is_file():
        print(path.name, round(path.stat().st_size / 1e6, 2), 'MB')
print('Peak RAM (reported by OS):', round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), 'GB')

## 14. Consuming v3

Tabular models read `train.parquet` and `test.parquet`; the hybrid reads those
tables plus `seq_hybrid_*.npy`. A listening-only LSTM reads the eight-channel
hybrid arrays. A transaction-only LSTM takes channels 8–19 from `seq_lstm_*.npy`;
the combined LSTM reads all twenty channels. Upload the two parquet files, four
arrays, metadata, parameters and audit to Kaggle as one v3 dataset. Readers should
check `dataset_version=3`, ordered channel names, shape, dtype and row hash.
See [Sherlock](SHERLOCK-%20KKBox%20Modelling.ipynb) for model fitting. The old
`seq_train.npy` contract is v2 and does not describe these arrays.